# Ironworks Offer Automation

An ironworks company receives customer enquiries via email, accompanied by
heterogeneous technical documentation (drawings, spec sheets, catalogues).
A domain expert manually reviews each request, translates specs into product
requirements, and prepares a formal offer.

This notebook shows how to automate that workflow with prompt-forge:

1. **Bootstrap** — convert existing PDF offers to structured JSON (one-time setup).
2. **Train** — teach the model to generate offers from emails + attachments.
3. **Infer** — generate a structured offer for any new customer request.

**Data layout expected:**

```
training_data/
    case_001/
        email.txt                  # customer enquiry email
        attachments_drawing.pdf    # technical drawing (variadic — 0..N files)
        attachments_spec.pdf       # spec sheet
        offer.pdf                  # ground truth offer (PDF — not loaded as a bundle role)
        expected_output.json       # ← created by the bootstrap step below
    case_002/
        ...
```

Attachment files must be named `attachments_*.pdf` so the loader picks them up
for the variadic `attachments` role and ignores `offer.pdf`.

## LLM client

Native PDF support requires the Azure OpenAI Responses API.

In [ ]:
import base64
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import AzureOpenAI

from prompt_forge import LLMMessage, LLMResponse, TextPart, FilePart

load_dotenv()


class AzureResponsesClient:
    """Azure OpenAI Responses API — supports native PDF and image inputs."""

    ENDPOINT   = "https://your-resource.openai.azure.com/"
    API_VERSION = "2025-03-01-preview"
    DEPLOYMENT = "gpt-4o"

    def __init__(self):
        self.client = AzureOpenAI(
            api_version=self.API_VERSION,
            azure_endpoint=self.ENDPOINT,
            api_key=os.environ["AZURE_OPENAI_API_KEY"],
        )

    def complete(self, messages: list[LLMMessage], **kwargs) -> LLMResponse:
        allowed = {"temperature", "max_tokens", "top_p"}
        oai_kwargs = {k: v for k, v in kwargs.items() if k in allowed}

        input_ = []
        for m in messages:
            if isinstance(m.content, str):
                input_.append({"role": m.role, "content": m.content})
            else:
                parts = []
                for part in m.content:
                    if isinstance(part, TextPart):
                        parts.append({"type": "input_text", "text": part.text})
                    elif isinstance(part, FilePart):
                        data = base64.b64encode(part.path.read_bytes()).decode()
                        mime = part.media_type or "application/octet-stream"
                        parts.append({
                            "type": "input_file",
                            "filename": part.path.name,
                            "file_data": f"data:{mime};base64,{data}",
                        })
                input_.append({"role": m.role, "content": parts})

        resp = self.client.responses.create(
            model=self.DEPLOYMENT, input=input_, **oai_kwargs
        )
        return LLMResponse(
            text=resp.output_text,
            usage={
                "input_tokens": resp.usage.input_tokens,
                "output_tokens": resp.usage.output_tokens,
            },
        )


llm = AzureResponsesClient()

## Step 0 — Bootstrap: extract JSON from PDF offers

We only have PDF offers as ground truth, but prompt-forge needs structured JSON
to run `JsonFieldEvaluator`. The bootstrap step extracts JSON from each PDF **once**
using a strong LLM and saves it as `expected_output.json`.

This is a one-time operation. After it runs you commit `expected_output.json`
alongside the source files and never need to re-run it.

> **Alternative:** skip this step and use `LLMJudgeEvaluator` instead — the judge
> reads the extracted PDF text and scores the model's output semantically.
> This avoids bootstrap but is ~2× more expensive per eval call and less precise.
> See the commented block at the end of the Training section.

In [ ]:
import json
import logging

from prompt_forge.file_loaders import get_default_loader

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(name)s | %(message)s")

DATA_DIR = Path("training_data")

BOOTSTRAP_SYSTEM = """\
You are an offer document parser for an ironworks company.
Read the provided offer PDF and extract its content into a JSON object.

Return ONLY valid JSON — no explanation, no markdown fences.

Required fields:
{
  "customer_name": string,
  "project_description": string,
  "line_items": [
    {
      "description": string,
      "quantity": number,
      "unit": string,
      "unit_price_eur": number,
      "total_price_eur": number,
      "notes": string | null
    }
  ],
  "total_price_eur": number,
  "lead_time_weeks": number,
  "validity_days": number,
  "delivery_terms": string,
  "payment_terms": string
}

If a field cannot be determined, use null.
Arrays must be valid JSON arrays; numbers must be numeric types.
"""


def extract_offer_json(offer_pdf: Path) -> dict:
    """Use the LLM to extract structured JSON from a PDF offer."""
    data = base64.b64encode(offer_pdf.read_bytes()).decode()
    messages = [
        LLMMessage(role="system", content=BOOTSTRAP_SYSTEM),
        LLMMessage(role="user", content=[
            TextPart(text="Extract the offer data from this PDF:"),
            FilePart(path=offer_pdf, media_type="application/pdf"),
        ]),
    ]
    response = llm.complete(messages, temperature=0.0)
    text = response.text.strip()
    # Strip markdown fences if present
    if text.startswith("```"):
        text = text.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    return json.loads(text)


def bootstrap_directory(data_dir: Path, overwrite: bool = False) -> int:
    """
    Iterate over all case subdirectories and extract offer JSON where missing.

    Args:
        data_dir: Root directory containing one subdirectory per case.
        overwrite: If True, re-extract even if expected_output.json already exists.

    Returns:
        Number of cases processed.
    """
    cases = sorted(p for p in data_dir.iterdir() if p.is_dir())
    processed = 0
    for case in cases:
        out_path = case / "expected_output.json"
        if out_path.exists() and not overwrite:
            logging.info(f"Skipping {case.name} — expected_output.json already exists")
            continue
        offer_pdf = case / "offer.pdf"
        if not offer_pdf.exists():
            logging.warning(f"Skipping {case.name} — offer.pdf not found")
            continue
        logging.info(f"Extracting {case.name}...")
        try:
            data = extract_offer_json(offer_pdf)
            out_path.write_text(json.dumps(data, indent=2, ensure_ascii=False))
            logging.info(f"  → {out_path} ({len(data)} top-level fields)")
            processed += 1
        except Exception as e:
            logging.error(f"  Failed: {e}")
    return processed


n = bootstrap_directory(DATA_DIR)
print(f"\nBootstrap complete — {n} expected_output.json files created.")

### Verify the extraction

Spot-check a few extracted JSONs before proceeding. The quality of the bootstrap
determines the quality of your training signal — fix any systematic extraction errors
(e.g. wrong currency, truncated line items) before training.

In [ ]:
# Spot-check: print one extracted offer
sample_case = sorted(DATA_DIR.iterdir())[0]
sample_json = json.loads((sample_case / "expected_output.json").read_text())
print(json.dumps(sample_json, indent=2, ensure_ascii=False))

## Step 1 — Project setup

In [ ]:
from prompt_forge import Project, TrainingConfig, train_val_split

PROJECT_DIR = ".projects/ironworks_offer"

project = Project("ironworks_offer", llm=llm, project_dir=PROJECT_DIR)

# email: the customer enquiry (one per case)
# attachments: technical docs — variadic (0..N PDFs per case, named attachments_*.pdf)
# expected_output: the JSON offer extracted in the bootstrap step
project.set_bundle_schema(
    email=".txt",
    attachments=".pdf",
    expected_output=".json",
    variadic=["attachments"],
)

project.set_context(
    "Ironworks company offer automation. "
    "Customers send enquiries with technical drawings and spec sheets. "
    "The system must translate technical requirements into a formal offer "
    "with itemised line items, pricing, lead time, and commercial terms. "
    "Prices are in EUR. Dimensions and weights are metric."
)

project.set_seed_prompt(
    "You are an expert ironworks offer specialist. "
    "Read the customer email and any attached technical documentation, "
    "then prepare a structured offer in JSON format."
)

# Declare the expected JSON output structure
project.set_output_schema({
    "customer_name": "string",
    "project_description": "string",
    "line_items": "array of {description, quantity, unit, unit_price_eur, total_price_eur, notes}",
    "total_price_eur": "number",
    "lead_time_weeks": "number",
    "validity_days": "number",
    "delivery_terms": "string",
    "payment_terms": "string",
})

n = project.add_examples_from_directory(DATA_DIR)
print(f"Loaded {n} examples")
print(project)

## Step 2 — Train

We use a 70/15/15 split: train / val / test.

The **validation set** drives early stopping and version selection.
The **test set** is held out entirely and evaluated once at the end
for an unbiased performance estimate.

`JsonFieldEvaluator` scores field-by-field and handles numeric rounding
and date normalisation automatically. `line_items` is compared as a whole
array — if the exact contents differ, that field scores 0. For a more
lenient semantic comparison of line items, switch to `LLMJudgeEvaluator`
(commented block below).

In [ ]:
from prompt_forge import train_val_split, JsonFieldEvaluator, LLMJudgeEvaluator

all_bundles = project.bundles.bundles

# Hold out test set first — never seen during training or model selection
train_val, test = train_val_split(all_bundles, val_ratio=0.15, seed=42)
train, val = train_val_split(train_val, val_ratio=0.15, seed=42)

print(f"Split: {len(train)} train / {len(val)} val / {len(test)} test")

# ── Evaluator choice ──────────────────────────────────────────────────────────
# Option A — field-by-field JSON comparison (fast, deterministic, free)
eval_strategy = JsonFieldEvaluator(
    numeric_tolerance=0.5,    # allow ±0.50 EUR rounding differences
    ignore_fields=["notes"],  # free-text notes are hard to match exactly
)

# Option B — LLM judge (semantic, handles paraphrased line items, costs tokens)
# eval_strategy = LLMJudgeEvaluator(
#     llm=llm,
#     task_description=(
#         "Ironworks offer document. The actual output is a JSON offer. "
#         "Evaluate accuracy of line items, pricing, lead time, and commercial terms."
#     ),
# )

In [ ]:
report = project.train(
    train,
    val_bundles=val,
    eval_strategy=eval_strategy,
    config=TrainingConfig(
        batch_size=4,          # small batches — each example is large (email + PDFs)
        max_iterations=20,
        patience=4,
        native_files=True,     # pass PDFs directly to the model
        inference_temperature=0.0,
        max_workers=4,         # concurrent eval calls (native files can't batch)
        seed=42,
    ),
)

print(f"\nFinal version : v{report.final_version}")
print(f"Final val score: {report.final_score:.3f}" if report.final_score else "Final val score: —")
print(f"Total tokens   : {report.total_tokens_used:,}")

## Step 3 — Inspect results

In [ ]:
for r in report:
    status = "✓" if r.improved else "✗"
    before = f"{r.score_before:.3f}" if r.score_before is not None else "—"
    after  = f"{r.score_after:.3f}"  if r.score_after  is not None else "—"
    tok    = f"[{r.tokens_used:,} tok]" if r.tokens_used else ""
    print(f"Iter {r.iteration:2d}: {before} → {after} {status}  {tok}")
    if r.val_example_scores:
        worst = min(r.val_example_scores, key=r.val_example_scores.get)
        print(f"         worst val example: {worst} ({r.val_example_scores[worst]:.3f})")

# Summarise what gaps remain — tells you which kinds of cases to add next
summary = report.aggregate_issues(llm)
if summary:
    print("\nRecurring issues:")
    print(summary)

In [ ]:
# Prompt version history
print("Prompt versions:")
for v in project.list_versions():
    score = f"score={v.eval_score:.3f}" if v.eval_score is not None else "no score"
    consolidation = " [CONSOLIDATION]" if v.metadata.get("consolidation") else ""
    entry = (v.training_log_entry or "")[:100]
    print(f"  v{v.version:03d}  {score}{consolidation}  {entry}")

## Step 4 — Test set evaluation

Evaluate the final prompt on the held-out test set.
This gives an unbiased estimate of real-world performance.

In [ ]:
from prompt_forge.bundle import is_output_role

agent = project.get_inference_agent(version=report.final_version, native_files=True)

test_results_raw = []
for bundle in test:
    actual = agent.run_bundle(bundle)
    contents = bundle.load_contents()
    expected = next(
        content.text for role, content in contents.items() if is_output_role(role)
    )
    test_results_raw.append((bundle.bundle_id, actual, expected))

test_eval = eval_strategy.evaluate_batch(test_results_raw)
print(f"Test score  : {test_eval.mean_score:.3f}")
print(f"Test pass rate: {test_eval.pass_rate:.0%}  ({sum(1 for r in test_eval.individual_results if r.passed)}/{len(test)} passed)")
print()
if test_eval.failed_examples:
    print("Failed examples:")
    for f in test_eval.failed_examples:
        print(f"  {f['bundle_id']}: {f['feedback'][:120]}")

## Step 5 — Production inference

At inference time, pass the customer email and any attachments.
The agent returns a structured JSON offer ready for review or downstream processing.

In [ ]:
# Single file input
result = agent.run(
    input_files={
        "email": "new_requests/customer_email.txt",
        "attachments": "new_requests/technical_drawing.pdf",
    }
)
print(json.dumps(result, indent=2, ensure_ascii=False))

In [ ]:
# Multiple attachments — build the dict dynamically
import glob

email_path = "new_requests/customer_email.txt"
attachment_paths = sorted(glob.glob("new_requests/attachments/*.pdf"))

input_files = {"email": email_path}
for i, p in enumerate(attachment_paths, 1):
    input_files[f"attachments_{i}"] = p

result = agent.run(input_files=input_files)
print(f"Customer    : {result.get('customer_name')}")
print(f"Total (EUR) : {result.get('total_price_eur'):,.2f}")
print(f"Lead time   : {result.get('lead_time_weeks')} weeks")
print(f"Line items  : {len(result.get('line_items', []))}")
print()
print(json.dumps(result, indent=2, ensure_ascii=False))